In [128]:
import pandas as pd
import numpy as np
import gzip
import matplotlib.pyplot as plt
import pickle
from scipy.stats import linregress, gmean
import os
import sys
from datetime import date
from sklearn.metrics import auc, f1_score
import random
from sklearn.decomposition import NMF, non_negative_factorization
import seaborn as sns
import time
from matplotlib.colors import LogNorm, Normalize
from mpl_toolkits.axes_grid1 import axes_grid
import pyreadr
import statistics as stats
import math
from tabulate import tabulate
from statsmodels.stats.contingency_tables import mcnemar
import re

In [14]:
#load npz file
mixture_all_cycles = []

#original mixture
fn = gzip.GzipFile('/home/ctrader/Hold/DHS/2018-06-08NC16_NNDSVD_Mixture.npy.gz', "r")
mixture_all_cycles.append(np.float32(np.load(fn)))

for i in range(1, 2):
    print(i)
    filename = '/home/ctrader/Hold/NMF/DHS/fill_in_multiple_at_a_time_saves/reconstructed_threshold_flip_booldhsarr_basis_mixture_to0thresh0.0_to1thresh1.0_cycle'+str(i)+'_CT20220323.npz'
    cycle_np_file = np.load(filename)
    mixture_all_cycles.append(cycle_np_file['new_decomp_mixture'])

mixture_all_cycles = np.array(mixture_all_cycles)

1


In [15]:
n_components, n_DHSs = mixture_all_cycles[0].shape

In [4]:
mixture_all_cycles.shape

(2, 16, 3591898)

In [17]:
DHS_metadata = pd.read_table('/home/ctrader/Hold/DHS/DHS_Index_and_Vocabulary_hg38_WM20190703.txt.gz', header=None)

col_names = DHS_metadata.iloc[0]
DHS_metadata = DHS_metadata[1:]
DHS_metadata.columns = col_names
DHS_metadata = DHS_metadata.reset_index()
DHS_metadata[['identifier']] = DHS_metadata[['identifier']].astype(str)
DHS_metadata = DHS_metadata.drop(columns=['index'])
DHS_metadata['mean_signal'] = DHS_metadata['mean_signal'].astype(float)

#top 10%
DHS_metadata['ind'] = DHS_metadata.index
DHS_metadata_sortby_mean_signal = DHS_metadata.sort_values(['mean_signal'], ascending=False)
top_10_percent_DHSs = DHS_metadata_sortby_mean_signal.iloc[:int(np.round(n_DHSs/10))]
top_10_percent_DHSs_inds = np.array(top_10_percent_DHSs['ind'])
top_10_percent_DHSs_identifiers = np.array(top_10_percent_DHSs['identifier'])

/home/ctrader/.conda/envs/lightfm/lib/python3.8/site-packages/IPython/core/interactiveshell.py:3441: DtypeWarning: Columns (1,2,3,4,5,6,7,8) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [6]:
pickle_in = open('/home/ctrader/Hold/NMF/DHS/gene_DHSs_all_dict_CT20220324.pickle', 'rb')
gene_DHSs_all_dict=pickle.load(pickle_in)

In [7]:
comps_for_heatmap = ['Tissue invariant','Stromal A', 'Primitive / embryonic', 'Stromal B', 'Lymphoid', 'Renal / cancer', 'Placental / trophoblast', 'Neural', 'Cardiac', 'Organ devel. / renal', 'Pulmonary devel.', 'Musculoskeletal', 'Digestive', 'Vascular / endothelial', 'Myeloid / erythroid', 'Cancer / epithelial']
comps_for_heatmap = np.array(comps_for_heatmap)

In [12]:
#load npz file
added_recon_all_cycles = []
removed_recon_all_cycles = []
added_flat_inds_all_cycles = []
removed_flat_inds_all_cycles = []

for i in range(1, 2):
    filename = '/home/ctrader/Hold/NMF/DHS/fill_in_multiple_at_a_time_saves/reconstructed_threshold_flip_added_removed_recon_flatinds_to0thresh0.0_to1thresh1.0_cycle'+str(i)+'_CT20220323.npz'
    cycle_np_file = np.load(filename)
    added_recon_all_cycles.append(cycle_np_file['added_recon'])
    removed_recon_all_cycles.append(cycle_np_file['removed_recon'])
    added_flat_inds_all_cycles.append(cycle_np_file['added_flat_inds'])
    removed_flat_inds_all_cycles.append(cycle_np_file['removed_flat_inds'])

In [198]:
removed_DHS_inds = removed_flat_inds_all_cycles[0]%n_DHSs
added_DHS_inds = added_flat_inds_all_cycles[0]%n_DHSs

removed_biosample_inds = removed_flat_inds_all_cycles[0]//n_DHSs
added_biosample_inds = added_flat_inds_all_cycles[0]//n_DHSs

In [199]:
#flip based and non-flip based DHS reassignments
changed_dhss_all_cycles_flipbased = []
changed_dhss_original_comp_flipbased = []
changed_dhss_new_comp_flipbased = []
changed_dhss_numflips_flipbased = []

changed_dhss_all_cycles_notflipbased = []
changed_dhss_original_comp_notflipbased = []
changed_dhss_new_comp_notflipbased = []

for i in range(1):
    changed_dhss_1_cycle_flipbased = []
    original_comp_flipbased = []
    new_comp_flipbased = []
    numflips_flipbased = []
    
    changed_dhss_1_cycle_notflipbased = []
    original_comp_notflipbased = []
    new_comp_notflipbased = []
    
    for comp1, comp0, dhs in zip(mixture_all_cycles_top_comp[i+1], 
                                 mixture_all_cycles_top_comp[i], 
                                 range(len(mixture_all_cycles_top_comp[0]))):
        if ((comp1!=comp0) and (~np.isnan(comp0)) and (~np.isnan(comp1))):
            comp0 = comp0.astype(int)
            comp1 = comp1.astype(int)
            if ((dhs in removed_DHS_inds) | (dhs in added_DHS_inds)):
                changed_dhss_1_cycle_flipbased.append(dhs)
                original_comp_flipbased.append(comp0)
                new_comp_flipbased.append(comp1)
                numflips_flipbased.append(len(np.where(removed_DHS_inds==dhs)[0]) + 
                                          len(np.where(added_DHS_inds==dhs)[0]))
            else:
                changed_dhss_1_cycle_notflipbased.append(dhs)
                original_comp_notflipbased.append(comp0)
                new_comp_notflipbased.append(comp1)
            
    changed_dhss_all_cycles_flipbased.append(np.array(changed_dhss_1_cycle_flipbased))
    changed_dhss_original_comp_flipbased.append(np.array(original_comp_flipbased))
    changed_dhss_new_comp_flipbased.append(np.array(new_comp_flipbased))
    changed_dhss_numflips_flipbased.append(np.array(numflips_flipbased))
    
    changed_dhss_all_cycles_notflipbased.append(np.array(changed_dhss_1_cycle_notflipbased))
    changed_dhss_original_comp_notflipbased.append(np.array(original_comp_notflipbased))
    changed_dhss_new_comp_notflipbased.append(np.array(new_comp_notflipbased))

**Find overlapping DHSs and GWAS SNPs**

In [2]:
#load data
data = []
with open("/home/ctrader/lightfm_data/GWAS_full_dataset/UKBB_all_traits_all_sig_associations_CT20220207.bed") as f:
    for line in f:
        data.append(line.strip().split())

#combine label elements
for row in data:
    row.append(' '.join(row[5:]))
    del row[5:-1]
    
#create dataframe
sig_df = pd.DataFrame(data, columns = ['Chromosome', 'p1', 'p2', 'Reference SNP cluster ID', 'pvalue', 'phenotype'])

In [133]:
sig_df['pvalue'] = sig_df['pvalue'].astype(float)

In [7]:
sig_df_groupbysnp = sig_df.groupby('Reference SNP cluster ID').first()
sig_df_groupbysnp_snp = np.array(sig_df_groupbysnp.index)
sig_df_groupbysnp_chrm = np.array(sig_df_groupbysnp['Chromosome'])
sig_df_groupbysnp_p1 = np.array(sig_df_groupbysnp['p1']).astype(int)
sig_df_groupbysnp_p2 = np.array(sig_df_groupbysnp['p2']).astype(int)

In [16]:
removed_DHS_inds = removed_flat_inds_all_cycles[0]%n_DHSs
added_DHS_inds = added_flat_inds_all_cycles[0]%n_DHSs

removed_biosample_inds = removed_flat_inds_all_cycles[0]//n_DHSs
added_biosample_inds = added_flat_inds_all_cycles[0]//n_DHSs

In [20]:
added_chrm = np.array(DHS_metadata.loc[added_DHS_inds]['seqname'])
added_start = np.array(DHS_metadata.loc[added_DHS_inds]['start'])
added_end = np.array(DHS_metadata.loc[added_DHS_inds]['end'])
added_identifier = np.array(DHS_metadata.loc[added_DHS_inds]['identifier'])

removed_chrm = np.array(DHS_metadata.loc[removed_DHS_inds]['seqname'])
removed_start = np.array(DHS_metadata.loc[removed_DHS_inds]['start'])
removed_end = np.array(DHS_metadata.loc[removed_DHS_inds]['end'])
removed_identifier = np.array(DHS_metadata.loc[removed_DHS_inds]['identifier'])

In [21]:
today = date.today()
bed_path_name = os.path.join('/home/ctrader/NMF_data/matching_DHSs_to_SNPs/', 'NMF_threshold_flip_01thresh_cycle1_added_call_DHSs_CT' + today.strftime("%Y%m%d") + '.bed')
f= open(bed_path_name, 'w')
for i in range(len(added_chrm)):
    f.write(added_chrm[i] + '\t' + str(added_start[i]) + '\t' + str(added_end[i]) + '\t' + str(added_identifier[i]) + '\t' + str(i) + '\n')
f.close()

In [22]:
today = date.today()
bed_path_name = os.path.join('/home/ctrader/NMF_data/matching_DHSs_to_SNPs/', 'NMF_threshold_flip_01thresh_cycle1_removed_call_DHSs_CT' + today.strftime("%Y%m%d") + '.bed')
f= open(bed_path_name, 'w')
for i in range(len(removed_chrm)):
    f.write(removed_chrm[i] + '\t' + str(removed_start[i]) + '\t' + str(removed_end[i]) + '\t' + str(removed_identifier[i]) + '\t' + str(i) + '\n')
f.close()

In [23]:
today = date.today()
bed_path_name = os.path.join('/home/ctrader/NMF_data/matching_DHSs_to_SNPs/', 'all_GWAS_snps_with_a_significant_association_CT' + today.strftime("%Y%m%d") + '.bed')
f= open(bed_path_name, 'w')
for i in range(len(sig_df_groupbysnp_chrm)):
    f.write(sig_df_groupbysnp_chrm[i] + '\t' + str(sig_df_groupbysnp_p1[i]) + '\t' + str(sig_df_groupbysnp_p2[i]) + '\t' + str(sig_df_groupbysnp_snp[i]) + '\n')
f.close()

In [80]:
#make snp mapped added DHS dict. Keys are DHSs that are added, values are mapped snps
added_snp_mapped = {}
with open("/home/ctrader/NMF_data/matching_DHSs_to_SNPs/NMF_threshold_flip_01thresh_cycle1_added_call_DHSs_mapped_to_sig_GWAS_snps_CT20220412.bed") as f:
    for line in f:
        dhs_data = re.split(';|\t|\|', line.strip())
        added_snp_mapped[added_DHS_inds[int(dhs_data[4])]]=np.array(dhs_data[8::4])

In [81]:
#make snp mapped removed DHS dict. Keys are DHSs that are removed, values are mapped snps
removed_snp_mapped = {}
with open("/home/ctrader/NMF_data/matching_DHSs_to_SNPs/NMF_threshold_flip_01thresh_cycle1_removed_call_DHSs_mapped_to_sig_GWAS_snps_CT20220412.bed") as f:
    for line in f:
        dhs_data = re.split(';|\t|\|', line.strip())
        removed_snp_mapped[removed_DHS_inds[int(dhs_data[4])]]=np.array(dhs_data[8::4])

In [88]:
#which of the added and removed call DHS inds map to at least 1 snp? 
added_DHS_inds_mapping_to_snp = np.array(list(added_snp_mapped.keys()))
removed_DHS_inds_mapping_to_snp = np.array(list(removed_snp_mapped.keys()))

In [125]:
len(added_DHS_inds_mapping_to_snp)

6124

In [134]:
sig_df['pvalue'][0]

3.55502e-08

In [154]:
snp_pvals_mean = sig_df.groupby('Reference SNP cluster ID')['pvalue'].apply(gmean)

/home/ctrader/.conda/envs/lightfm/lib/python3.8/site-packages/scipy/stats/stats.py:275: RuntimeWarning: divide by zero encountered in log
  log_a = np.log(np.array(a, dtype=dtype))


In [158]:
sig_df

2042473